In [ ]:
# Linux에서 시스템 오디오 녹음 설정
# 필수 패키지 설치 (처음 한 번만 실행):
# !sudo apt update
# !sudo apt install -y build-essential libsndfile1 portaudio19-dev libportaudio2 ffmpeg pulseaudio-utils

# Python 패키지 설치:
%pip install pyaudio soundfile pydub -q

In [1]:
# PulseAudio/PipeWire 모니터 장치 확인 (웹 브라우저 소리 녹음용)
import subprocess

print("=" * 70)
print("🎙️  웹 브라우저 소리 녹음을 위한 PulseAudio 모니터 장치 확인")
print("=" * 70)

try:
    result = subprocess.run(['pactl', 'list', 'short', 'sources'], 
                          capture_output=True, text=True, check=True)
    print("\n사용 가능한 입력 장치 (Monitor):")
    print(result.stdout)
    
    # monitor 장치 자동 선택
    lines = result.stdout.strip().split('\n')
    monitor_device = None
    for line in lines:
        if 'monitor' in line.lower():
            parts = line.split()
            if len(parts) > 0:
                monitor_device = parts[0]
                device_name = ' '.join(parts[1:])
                print(f"✅ 자동 선택된 모니터 장치: {monitor_device} - {device_name}")
                break
    
    if not monitor_device:
        print("⚠️  모니터 장치를 찾을 수 없습니다.")
        print("💡 팁: pactl load-module module-loopback latency_msec=1 sink=null 로 생성 가능")
        
except Exception as e:
    print(f"❌ 오류: {e}")

print("=" * 70)

🎙️  웹 브라우저 소리 녹음을 위한 PulseAudio 모니터 장치 확인

사용 가능한 입력 장치 (Monitor):
52	alsa_output.pci-0000_03_00.6.analog-stereo.monitor	PipeWire	s32le 2ch 48000Hz	SUSPENDED
53	alsa_input.pci-0000_03_00.6.analog-stereo	PipeWire	s32le 2ch 48000Hz	SUSPENDED

✅ 자동 선택된 모니터 장치: 52 - alsa_output.pci-0000_03_00.6.analog-stereo.monitor PipeWire s32le 2ch 48000Hz SUSPENDED


In [2]:
import pyaudio
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import os
import threading
from queue import Queue
import subprocess

# ============================================================
# 설정
# ============================================================
THRESHOLD_PEAK = 600          # 녹음 시작 음량 임계값
SILENCE_LIMIT = 3.0           # 녹음 종료 무음 시간 (초)
PAUSE_TOLERANCE_SECONDS = 0.5 # 짧은 무음 허용 시간 (초)
MIN_RECORDING_DURATION = 0.5  # 저장할 최소 녹음 길이 (초)
MP3_BITRATE = "128k"          # MP3 비트레이트
CHUNK = 1024                  # 오디오 청크 크기
OUTPUT_FORMAT = "mp3"         # 출력 포맷 (mp3 또는 flac)

# ============================================================
# 1. 웹 브라우저 소리 녹음을 위한 PulseAudio 모니터 장치 감지
# ============================================================
def find_monitor_device():
    """PulseAudio 모니터 장치를 자동으로 찾기"""
    try:
        result = subprocess.run(['pactl', 'list', 'short', 'sources'], 
                              capture_output=True, text=True, check=True)
        for line in result.stdout.strip().split('\n'):
            if 'monitor' in line.lower() and 'alsa_output' in line:
                parts = line.split()
                return parts[0], ' '.join(parts[1:])
    except Exception as e:
        print(f"⚠️  pactl 실행 오류: {e}")
    return None, None

monitor_id, monitor_name = find_monitor_device()

# ============================================================
# 2. PyAudio를 통해 사용 가능한 입력 장치 찾기
# ============================================================
audio = pyaudio.PyAudio()
INPUT_DEVICE_INDEX = None

print("\n" + "=" * 70)
print("🎤 Linux 시스템 오디오 녹음 프로그램")
print("=" * 70)

if monitor_name:
    print(f"\n✅ 웹 브라우저 소리 모니터 장치 감지됨: {monitor_name}")
    print(f"   (PulseAudio 장치 ID: {monitor_id})")
else:
    print("\n⚠️  웹 브라우저 소리 모니터 장치를 찾을 수 없습니다.")
    print("   다음 명령어로 loopback 장치를 생성할 수 있습니다:")
    print("   pactl load-module module-loopback latency_msec=1 sink=null")

# PyAudio 장치 확인
print("\n📋 PyAudio 입력 장치 목록:")
for i in range(audio.get_device_count()):
    try:
        info = audio.get_device_info_by_index(i)
        if info['maxInputChannels'] > 0:
            name = info.get('name', '')
            is_default = '(기본)' if i == audio.get_default_input_device_info()['index'] else ''
            print(f"  INDEX {i}: {name} {is_default}")
            
            # pipewire, pulse, default 중 첫 번째 선택
            if INPUT_DEVICE_INDEX is None and any(x in name.lower() for x in ['pipewire', 'pulse', 'default']):
                INPUT_DEVICE_INDEX = i
    except:
        pass

if INPUT_DEVICE_INDEX is None:
    INPUT_DEVICE_INDEX = 0

device_info = audio.get_device_info_by_index(INPUT_DEVICE_INDEX)
RATE = int(device_info['defaultSampleRate'])
CHANNELS = min(2, device_info['maxInputChannels'])
FORMAT = pyaudio.paInt16

print(f"\n🔧 선택된 입력 장치: INDEX {INPUT_DEVICE_INDEX} - {device_info['name']}")
print(f"   샘플레이트: {RATE}Hz | 채널: {CHANNELS} | 포맷: 16bit PCM")
print(f"\n📊 녹음 설정:")
print(f"   - 시작 임계값: Peak > {THRESHOLD_PEAK}")
print(f"   - 종료 무음: {SILENCE_LIMIT}초 이상")
print(f"   - 최소 길이: {MIN_RECORDING_DURATION}초")
print(f"   - 출력 포맷: {OUTPUT_FORMAT.upper()} ({MP3_BITRATE})")

# ============================================================
# 3. 오디오 청크 관련 계산
# ============================================================
CHUNK_DURATION = CHUNK / RATE
PAUSE_TOLERANCE_CHUNKS = int(PAUSE_TOLERANCE_SECONDS / CHUNK_DURATION)

# ============================================================
# 4. 백그라운드 MP3 변환 스레드
# ============================================================
conversion_queue = Queue()
keep_converting = True

def mp3_converter_thread():
    """FLAC 파일을 MP3로 변환하는 백그라운드 스레드"""
    while keep_converting:
        try:
            flac_file, mp3_file = conversion_queue.get(timeout=1)
            
            try:
                print(f"\n🔄 MP3 변환 중: {os.path.basename(flac_file)}")
                audio_data = AudioSegment.from_file(flac_file, format="flac")
                audio_data.export(mp3_file, format="mp3", bitrate=MP3_BITRATE)
                os.remove(flac_file)
                file_size = os.path.getsize(mp3_file) / 1024
                print(f"✅ 완료: {os.path.basename(mp3_file)} ({file_size:.1f}KB)")
            except Exception as e:
                print(f"❌ 변환 실패: {e}")
                
            conversion_queue.task_done()
        except:
            continue

converter = threading.Thread(target=mp3_converter_thread, daemon=True)
converter.start()

# ============================================================
# 5. PyAudio 스트림 설정
# ============================================================
try:
    stream = audio.open(
        format=FORMAT, 
        channels=CHANNELS,
        rate=RATE, 
        input=True,
        input_device_index=INPUT_DEVICE_INDEX,
        frames_per_buffer=CHUNK
    )
    print("\n✅ 오디오 스트림 설정 완료")
except Exception as e:
    print(f"\n❌ 오디오 스트림 오류: {e}")
    audio.terminate()
    exit(1)

# ============================================================
# 6. 녹음 메인 루프
# ============================================================
recording = False
frames = []
silence_counter = 0
counter = 1
pause_chunk_count = 0

print(f"\n🎵 시작 준비 완료!")
print(f"⏹️  웹 브라우저에서 소리를 재생하면 자동 녹음됩니다.")
print(f"💻 종료: Ctrl+C 입력\n")
print("=" * 70 + "\n")

try:
    while True:
        # 오디오 데이터 읽기
        raw_data = stream.read(CHUNK, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        volume_peak = np.max(np.abs(data))
        
        # 상태 표시
        status = "🔴 녹음중" if recording else "⏳ 대기"
        queue_size = conversion_queue.qsize()
        print(f"{status} | Peak: {volume_peak:5d} | 변환 대기: {queue_size}", end='\r')
        
        # ==================== 녹음 시작 ====================
        if volume_peak > THRESHOLD_PEAK and not recording:
            recording = True
            frames = []
            silence_counter = 0
            pause_chunk_count = 0
            print(f"\n✅ 녹음 시작: record_{counter}.{OUTPUT_FORMAT} (Peak: {volume_peak})")
        
        # ==================== 녹음 중 ====================
        if recording:
            # 임계값 이상: 일반 프레임 추가
            if volume_peak > THRESHOLD_PEAK:
                silence_counter = 0
                pause_chunk_count = 0
                frames.append(data)
            
            # 임계값 이하: 짧은 무음은 허용
            elif pause_chunk_count < PAUSE_TOLERANCE_CHUNKS:
                pause_chunk_count += 1
                frames.append(data)
            
            # 긴 무음: 녹음 종료
            else:
                silence_counter += CHUNK_DURATION
                frames.append(data)
        
        # ==================== 녹음 종료 ====================
        if recording and silence_counter > SILENCE_LIMIT:
            # 마지막 무음 부분 제거
            frames_to_remove = int((PAUSE_TOLERANCE_SECONDS + SILENCE_LIMIT) / CHUNK_DURATION)
            
            if len(frames) > frames_to_remove:
                data_to_save = np.concatenate(frames[:-frames_to_remove])
            else:
                data_to_save = np.concatenate(frames)
            
            # 스테레오 reshape
            if CHANNELS == 2:
                data_to_save = data_to_save.reshape(-1, 2)
            
            duration = len(data_to_save) / RATE
            
            # 최소 길이 체크 후 저장
            if duration > MIN_RECORDING_DURATION:
                if OUTPUT_FORMAT == "mp3":
                    flac_file = f"record_{counter}_temp.flac"
                    mp3_file = f"record_{counter}.mp3"
                    sf.write(flac_file, data_to_save, RATE, subtype='PCM_16')
                    conversion_queue.put((flac_file, mp3_file))
                else:
                    flac_file = f"record_{counter}.flac"
                    sf.write(flac_file, data_to_save, RATE, subtype='PCM_16')
                
                print(f"✔️  저장됨: record_{counter}.{OUTPUT_FORMAT} ({duration:.2f}초)")
            else:
                print(f"⚠️  너무 짧음 ({duration:.2f}초 < {MIN_RECORDING_DURATION}초) - 무시됨")
            
            # 상태 초기화
            counter += 1
            recording = False
            frames = []
            silence_counter = 0
            pause_chunk_count = 0

except KeyboardInterrupt:
    print("\n\n⏹️  녹음 중지됨")
    print("⏳ 진행 중인 MP3 변환 완료 대기 중...")
    keep_converting = False
    conversion_queue.join()
    
finally:
    if stream.is_active():
        stream.stop_stream()
        stream.close()
    audio.terminate()
    print("\n✅ 프로그램 종료")
    print("=" * 70)

ALSA lib pcm_dsnoop.c:567:(snd_pcm_dsnoop_open) unable to open slave
ALSA lib pcm_dmix.c:1000:(snd_pcm_dmix_open) unable to open slave
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
Cannot connect to server socket err = No such file or directory
Cannot connect to server request channel
jack server is not running or cannot be started
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
Cannot connect to server socket err = No such file or directory
Cannot connect to server request channel
jack server is not running or cannot be started
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
JackShmReadWritePtr::~JackShmReadWritePtr - Init not done for -1, skipping unlock
ALSA lib pcm


🎤 Linux 시스템 오디오 녹음 프로그램

✅ 웹 브라우저 소리 모니터 장치 감지됨: alsa_output.pci-0000_03_00.6.analog-stereo.monitor PipeWire s32le 2ch 48000Hz SUSPENDED
   (PulseAudio 장치 ID: 52)

📋 PyAudio 입력 장치 목록:
  INDEX 1: HD-Audio Generic: CX8070 Analog (hw:1,0) 
  INDEX 3: pipewire 
  INDEX 4: pulse 
  INDEX 5: default (기본)

🔧 선택된 입력 장치: INDEX 3 - pipewire
   샘플레이트: 44100Hz | 채널: 2 | 포맷: 16bit PCM

📊 녹음 설정:
   - 시작 임계값: Peak > 600
   - 종료 무음: 3.0초 이상
   - 최소 길이: 0.5초
   - 출력 포맷: MP3 (128k)

✅ 오디오 스트림 설정 완료

🎵 시작 준비 완료!
⏹️  웹 브라우저에서 소리를 재생하면 자동 녹음됩니다.
💻 종료: Ctrl+C 입력


⏳ 대기 | Peak:     0 | 변환 대기: 0

⏹️  녹음 중지됨
⏳ 진행 중인 MP3 변환 완료 대기 중...

✅ 프로그램 종료
⏳ 대기 | Peak:     0 | 변환 대기: 0

⏹️  녹음 중지됨
⏳ 진행 중인 MP3 변환 완료 대기 중...

✅ 프로그램 종료


In [18]:
import pyaudio
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import os
import threading
from queue import Queue
import subprocess
from collections import deque

# ============================================================
# 설정
# ============================================================
THRESHOLD_PEAK = None  # None이면 자동 계산 (임계값 사전 측정)
AUTO_THRESHOLD_CALIBRATION = 3  # 캘리브레이션 시간 (초)
THRESHOLD_MULTIPLIER = 3  # 배경음 * 배수 = 감지 임계값

SILENCE_LIMIT = 3.0           # 녹음 종료 무음 시간 (초)
PAUSE_TOLERANCE_SECONDS = 0.5 # 짧은 무음 허용 시간 (초)
MIN_RECORDING_DURATION = 0.5  # 저장할 최소 녹음 길이 (초)
MP3_BITRATE = "128k"          # MP3 비트레이트
CHUNK = 1024                  # 오디오 청크 크기
OUTPUT_FORMAT = "mp3"         # 출력 포맷 (mp3 또는 flac)

# ============================================================
# 1. 웹 브라우저 소리 녹음을 위한 PulseAudio 모니터 장치 감지
# ============================================================
def find_monitor_device():
    """PulseAudio 모니터 장치를 자동으로 찾기"""
    try:
        result = subprocess.run(['pactl', 'list', 'short', 'sources'], 
                              capture_output=True, text=True, check=True)
        for line in result.stdout.strip().split('\n'):
            if 'monitor' in line.lower() and 'alsa_output' in line:
                parts = line.split()
                return parts[0], ' '.join(parts[1:])
    except Exception as e:
        print(f"⚠️  pactl 실행 오류: {e}")
    return None, None

monitor_id, monitor_name = find_monitor_device()

# ============================================================
# 2. PyAudio를 통해 사용 가능한 입력 장치 찾기
# ============================================================
audio = pyaudio.PyAudio()
INPUT_DEVICE_INDEX = None

print("\n" + "=" * 70)
print("🎤 Linux 시스템 오디오 녹음 프로그램")
print("=" * 70)

if monitor_name:
    print(f"\n✅ 웹 브라우저 소리 모니터 장치 감지됨: {monitor_name}")
    print(f"   (PulseAudio 장치 ID: {monitor_id})")
else:
    print("\n⚠️  웹 브라우저 소리 모니터 장치를 찾을 수 없습니다.")
    print("   다음 명령어로 loopback 장치를 생성할 수 있습니다:")
    print("   pactl load-module module-loopback latency_msec=1 sink=null")

# PyAudio 장치 확인
print("\n📋 PyAudio 입력 장치 목록:")
for i in range(audio.get_device_count()):
    try:
        info = audio.get_device_info_by_index(i)
        if info['maxInputChannels'] > 0:
            name = info.get('name', '')
            is_default = '(기본)' if i == audio.get_default_input_device_info()['index'] else ''
            print(f"  INDEX {i}: {name} {is_default}")
            
            # pipewire, pulse, default 중 첫 번째 선택
            if INPUT_DEVICE_INDEX is None and any(x in name.lower() for x in ['pipewire', 'pulse', 'default']):
                INPUT_DEVICE_INDEX = i
    except:
        pass

if INPUT_DEVICE_INDEX is None:
    INPUT_DEVICE_INDEX = 0

device_info = audio.get_device_info_by_index(INPUT_DEVICE_INDEX)
RATE = int(device_info['defaultSampleRate'])
CHANNELS = min(2, device_info['maxInputChannels'])
FORMAT = pyaudio.paInt16

print(f"\n🔧 선택된 입력 장치: INDEX {INPUT_DEVICE_INDEX} - {device_info['name']}")
print(f"   샘플레이트: {RATE}Hz | 채널: {CHANNELS} | 포맷: 16bit PCM")

# ============================================================
# 3. 오디오 청크 관련 계산
# ============================================================
CHUNK_DURATION = CHUNK / RATE
PAUSE_TOLERANCE_CHUNKS = int(PAUSE_TOLERANCE_SECONDS / CHUNK_DURATION)

# ============================================================
# 4. PyAudio 스트림 설정
# ============================================================
try:
    stream = audio.open(
        format=FORMAT, 
        channels=CHANNELS,
        rate=RATE, 
        input=True,
        input_device_index=INPUT_DEVICE_INDEX,
        frames_per_buffer=CHUNK
    )
    print("\n✅ 오디오 스트림 설정 완료")
except Exception as e:
    print(f"\n❌ 오디오 스트림 오류: {e}")
    audio.terminate()
    exit(1)

# ============================================================
# 5. 배경 노이즈 레벨 자동 캘리브레이션
# ============================================================
if THRESHOLD_PEAK is None:
    print(f"\n🔍 배경 노이즈 측정 중 ({AUTO_THRESHOLD_CALIBRATION}초)...")
    print("   ⚠️  조용한 상태를 유지하세요!")
    
    background_levels = deque(maxlen=int(RATE * AUTO_THRESHOLD_CALIBRATION / CHUNK))
    
    for _ in range(int(RATE * AUTO_THRESHOLD_CALIBRATION / CHUNK)):
        raw_data = stream.read(CHUNK, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        peak = np.max(np.abs(data))
        background_levels.append(peak)
    
    avg_background = np.mean(list(background_levels))
    max_background = np.max(list(background_levels))
    
    THRESHOLD_PEAK = int(max_background * THRESHOLD_MULTIPLIER)
    
    print(f"\n📊 캘리브레이션 결과:")
    print(f"   - 평균 배경음: {avg_background:.0f}")
    print(f"   - 최대 배경음: {max_background:.0f}")
    print(f"   - 설정된 임계값: {THRESHOLD_PEAK}")

print(f"\n📊 녹음 설정:")
print(f"   - 시작 임계값: Peak > {THRESHOLD_PEAK}")
print(f"   - 종료 무음: {SILENCE_LIMIT}초 이상")
print(f"   - 최소 길이: {MIN_RECORDING_DURATION}초")
print(f"   - 출력 포맷: {OUTPUT_FORMAT.upper()} ({MP3_BITRATE})")

# ============================================================
# 6. 백그라운드 MP3 변환 스레드
# ============================================================
conversion_queue = Queue()
keep_converting = True

def mp3_converter_thread():
    """FLAC 파일을 MP3로 변환하는 백그라운드 스레드"""
    while keep_converting:
        try:
            flac_file, mp3_file = conversion_queue.get(timeout=1)
            
            try:
                print(f"\n🔄 MP3 변환 중: {os.path.basename(flac_file)}")
                audio_data = AudioSegment.from_file(flac_file, format="flac")
                audio_data.export(mp3_file, format="mp3", bitrate=MP3_BITRATE)
                os.remove(flac_file)
                file_size = os.path.getsize(mp3_file) / 1024
                print(f"✅ 완료: {os.path.basename(mp3_file)} ({file_size:.1f}KB)")
            except Exception as e:
                print(f"❌ 변환 실패: {e}")
                
            conversion_queue.task_done()
        except:
            continue

converter = threading.Thread(target=mp3_converter_thread, daemon=True)
converter.start()

# ============================================================
# 7. 녹음 메인 루프
# ============================================================
recording = False
frames = []
silence_counter = 0
counter = 1
pause_chunk_count = 0

print(f"\n🎵 시작 준비 완료!")
print(f"⏹️  웹 브라우저에서 소리를 재생하면 자동 녹음됩니다.")
print(f"💻 종료: Ctrl+C 입력\n")
print("=" * 70 + "\n")

try:
    while True:
        # 오디오 데이터 읽기
        raw_data = stream.read(CHUNK, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        volume_peak = np.max(np.abs(data))
        
        # 상태 표시
        status = "🔴 녹음중" if recording else "⏳ 대기"
        queue_size = conversion_queue.qsize()
        print(f"{status} | Peak: {volume_peak:5d} | 임계값: {THRESHOLD_PEAK} | 변환 대기: {queue_size}", end='\r')
        
        # ==================== 녹음 시작 ====================
        if volume_peak > THRESHOLD_PEAK and not recording:
            recording = True
            frames = []
            silence_counter = 0
            pause_chunk_count = 0
            print(f"\n✅ 녹음 시작: record_{counter}.{OUTPUT_FORMAT} (Peak: {volume_peak})")
        
        # ==================== 녹음 중 ====================
        if recording:
            # 임계값 이상: 일반 프레임 추가
            if volume_peak > THRESHOLD_PEAK:
                silence_counter = 0
                pause_chunk_count = 0
                frames.append(data)
            
            # 임계값 이하: 짧은 무음은 허용
            elif pause_chunk_count < PAUSE_TOLERANCE_CHUNKS:
                pause_chunk_count += 1
                frames.append(data)
            
            # 긴 무음: 녹음 종료
            else:
                silence_counter += CHUNK_DURATION
                frames.append(data)
        
        # ==================== 녹음 종료 ====================
        if recording and silence_counter > SILENCE_LIMIT:
            # 마지막 무음 부분 제거
            frames_to_remove = int((PAUSE_TOLERANCE_SECONDS + SILENCE_LIMIT) / CHUNK_DURATION)
            
            if len(frames) > frames_to_remove:
                data_to_save = np.concatenate(frames[:-frames_to_remove])
            else:
                data_to_save = np.concatenate(frames)
            
            # 스테레오 reshape
            if CHANNELS == 2:
                data_to_save = data_to_save.reshape(-1, 2)
            
            duration = len(data_to_save) / RATE
            
            # 최소 길이 체크 후 저장
            if duration > MIN_RECORDING_DURATION:
                if OUTPUT_FORMAT == "mp3":
                    flac_file = f"record_{counter}_temp.flac"
                    mp3_file = f"record_{counter}.mp3"
                    sf.write(flac_file, data_to_save, RATE, subtype='PCM_16')
                    conversion_queue.put((flac_file, mp3_file))
                else:
                    flac_file = f"record_{counter}.flac"
                    sf.write(flac_file, data_to_save, RATE, subtype='PCM_16')
                
                print(f"✔️  저장됨: record_{counter}.{OUTPUT_FORMAT} ({duration:.2f}초)")
            else:
                print(f"⚠️  너무 짧음 ({duration:.2f}초 < {MIN_RECORDING_DURATION}초) - 무시됨")
            
            # 상태 초기화
            counter += 1
            recording = False
            frames = []
            silence_counter = 0
            pause_chunk_count = 0

except KeyboardInterrupt:
    print("\n\n⏹️  녹음 중지됨")
    print("⏳ 진행 중인 MP3 변환 완료 대기 중...")
    keep_converting = False
    conversion_queue.join()
    
finally:
    if stream.is_active():
        stream.stop_stream()
        stream.close()
    audio.terminate()
    print("\n✅ 프로그램 종료")
    print("=" * 70)


🎤 Linux 시스템 오디오 녹음 프로그램

✅ 웹 브라우저 소리 모니터 장치 감지됨: alsa_output.pci-0000_03_00.6.analog-stereo.monitor PipeWire s32le 2ch 48000Hz SUSPENDED
   (PulseAudio 장치 ID: 52)

📋 PyAudio 입력 장치 목록:
  INDEX 1: HD-Audio Generic: CX8070 Analog (hw:1,0) 
  INDEX 3: pipewire 
  INDEX 4: pulse 
  INDEX 5: default (기본)

🔧 선택된 입력 장치: INDEX 3 - pipewire
   샘플레이트: 44100Hz | 채널: 2 | 포맷: 16bit PCM

✅ 오디오 스트림 설정 완료

🔍 배경 노이즈 측정 중 (3초)...
   ⚠️  조용한 상태를 유지하세요!

📊 캘리브레이션 결과:
   - 평균 배경음: 0
   - 최대 배경음: 0
   - 설정된 임계값: 0

📊 녹음 설정:
   - 시작 임계값: Peak > 0
   - 종료 무음: 3.0초 이상
   - 최소 길이: 0.5초
   - 출력 포맷: MP3 (128k)

🎵 시작 준비 완료!
⏹️  웹 브라우저에서 소리를 재생하면 자동 녹음됩니다.
💻 종료: Ctrl+C 입력


⏳ 대기 | Peak:     0 | 임계값: 0 | 변환 대기: 0

⏹️  녹음 중지됨
⏳ 진행 중인 MP3 변환 완료 대기 중...

✅ 프로그램 종료


In [19]:
import pyaudio
import numpy as np

# ============================================================
# 🔍 진단: 현재 입력 장치 상태 확인
# ============================================================
print("=" * 70)
print("🔍 오디오 입력 장치 진단")
print("=" * 70)

audio = pyaudio.PyAudio()

print("\n📋 모든 입력 가능 장치:")
input_devices = []
for i in range(audio.get_device_count()):
    try:
        info = audio.get_device_info_by_index(i)
        if info['maxInputChannels'] > 0:
            input_devices.append((i, info['name']))
            is_default = '✓ 기본' if i == audio.get_default_input_device_info()['index'] else ''
            print(f"\n  [{i}] {info['name']} {is_default}")
            print(f"      입력 채널: {info['maxInputChannels']}")
            print(f"      출력 채널: {info['maxOutputChannels']}")
            print(f"      샘플레이트: {info['defaultSampleRate']}Hz")
    except:
        pass

# ============================================================
# 테스트: Peak 값 모니터링
# ============================================================
print("\n" + "=" * 70)
print("🎤 현재 입력 신호 모니터링 (10초)")
print("=" * 70)
print("\n💡 웹 브라우저에서 큰 소리를 재생해주세요!")
print("   (동시에 아래 Peak 값을 확인하세요)\n")

# 현재 설정된 장치로 테스트
try:
    device_info = audio.get_device_info_by_index(INPUT_DEVICE_INDEX)
    test_rate = int(device_info['defaultSampleRate'])
    test_channels = min(2, device_info['maxInputChannels'])
    
    test_stream = audio.open(
        format=pyaudio.paInt16,
        channels=test_channels,
        rate=test_rate,
        input=True,
        input_device_index=INPUT_DEVICE_INDEX,
        frames_per_buffer=1024
    )
    
    print(f"테스트 장치: INDEX {INPUT_DEVICE_INDEX} - {device_info['name']}\n")
    
    peaks = []
    for i in range(100):  # 약 1초 (100 * 1024 / 48000 ≈ 2초)
        raw_data = test_stream.read(1024, exception_on_overflow=False)
        data = np.frombuffer(raw_data, dtype=np.int16)
        peak = np.max(np.abs(data))
        peaks.append(peak)
        
        bar_length = int(peak / 32767 * 50)
        bar = '█' * bar_length + '░' * (50 - bar_length)
        print(f"  [{bar}] Peak: {peak:6d}", end='\r')
    
    test_stream.stop_stream()
    test_stream.close()
    
    avg_peak = np.mean(peaks)
    max_peak = np.max(peaks)
    
    print(f"\n\n📊 측정 결과:")
    print(f"   - 평균 Peak: {avg_peak:.0f}")
    print(f"   - 최대 Peak: {max_peak:.0f}")
    print(f"   - 현재 임계값: {THRESHOLD_PEAK}")
    
    if max_peak < THRESHOLD_PEAK * 0.5:
        print(f"\n⚠️  경고: 입력 신호가 매우 약합니다!")
        print(f"   최대 Peak({max_peak:.0f}) < 임계값({THRESHOLD_PEAK})의 절반")
        print(f"\n💡 해결 방법:")
        print(f"   1. 웹 브라우저 음량 확인")
        print(f"   2. 시스템 입력 레벨 조정: alsamixer")
        print(f"   3. 다른 입력 장치 시도 (아래 참조)")
    else:
        print(f"\n✅ 입력 신호 정상")
        
except Exception as e:
    print(f"❌ 테스트 실패: {e}")

audio.terminate()

print("\n" + "=" * 70)
print("💡 다른 입력 장치 시도하기")
print("=" * 70)
print("""
다음과 같이 수정하여 4번 셀을 다시 실행하세요:

INPUT_DEVICE_INDEX = 3  # 또는 다른 번호

또는 자동으로 가장 신호가 강한 장치를 찾으려면:
""")

# 모든 장치 테스트
print("\n모든 입력 장치의 Peak 값 측정:\n")

audio = pyaudio.PyAudio()
device_peaks = {}

for i, name in input_devices:
    try:
        test_stream = audio.open(
            format=pyaudio.paInt16,
            channels=1,
            rate=48000,
            input=True,
            input_device_index=i,
            frames_per_buffer=1024
        )
        
        peaks = []
        for _ in range(50):
            raw_data = test_stream.read(1024, exception_on_overflow=False)
            data = np.frombuffer(raw_data, dtype=np.int16)
            peaks.append(np.max(np.abs(data)))
        
        test_stream.stop_stream()
        test_stream.close()
        
        max_peak = np.max(peaks)
        device_peaks[i] = max_peak
        
        marker = "⭐ 추천" if max_peak == max(device_peaks.values()) else ""
        print(f"  [{i}] {name}: Max Peak = {max_peak:6d} {marker}")
        
    except:
        print(f"  [{i}] {name}: 테스트 불가")

audio.terminate()

if device_peaks:
    best_device = max(device_peaks, key=device_peaks.get)
    print(f"\n✅ 추천 장치: INDEX {best_device}")
    print(f"   INPUT_DEVICE_INDEX = {best_device} 로 설정 후 재시도하세요")

🔍 오디오 입력 장치 진단

📋 모든 입력 가능 장치:

  [1] HD-Audio Generic: CX8070 Analog (hw:1,0) 
      입력 채널: 2
      출력 채널: 2
      샘플레이트: 48000.0Hz

  [3] pipewire 
      입력 채널: 64
      출력 채널: 64
      샘플레이트: 44100.0Hz

  [4] pulse 
      입력 채널: 32
      출력 채널: 32
      샘플레이트: 44100.0Hz

  [5] default ✓ 기본
      입력 채널: 64
      출력 채널: 64
      샘플레이트: 44100.0Hz

🎤 현재 입력 신호 모니터링 (10초)

💡 웹 브라우저에서 큰 소리를 재생해주세요!
   (동시에 아래 Peak 값을 확인하세요)

테스트 장치: INDEX 3 - pipewire

  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] Peak:      0

📊 측정 결과:
   - 평균 Peak: 0
   - 최대 Peak: 0
   - 현재 임계값: 0

✅ 입력 신호 정상

💡 다른 입력 장치 시도하기

다음과 같이 수정하여 4번 셀을 다시 실행하세요:

INPUT_DEVICE_INDEX = 3  # 또는 다른 번호

또는 자동으로 가장 신호가 강한 장치를 찾으려면:


모든 입력 장치의 Peak 값 측정:

  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] Peak:      0

📊 측정 결과:
   - 평균 Peak: 0
   - 최대 Peak: 0
   - 현재 임계값: 0

✅ 입력 신호 정상

💡 다른 입력 장치 시도하기

다음과 같이 수정하여 4번 셀을 다시 실행하세요:

INPUT_DEVICE_INDEX = 3  # 또는 다른 번호

또는 자동으로 가장 신호가 강한 장치를 찾으려면:


모든 입력 장치의 Peak 값 측정:

  [

Expression 'ret' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 1736
Expression 'AlsaOpen( &alsaApi->baseHostApiRep, params, streamDir, &self->pcm )' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 1904
Expression 'PaAlsaStreamComponent_Initialize( &self->capture, alsaApi, inParams, StreamDirection_In, NULL != callback )' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 2171
Expression 'PaAlsaStream_Initialize( stream, alsaHostApi, inputParameters, outputParameters, sampleRate, framesPerBuffer, callback, streamFlags, userData )' failed in 'src/hostapi/alsa/pa_linux_alsa.c', line: 2839


  [3] pipewire: Max Peak =      0 ⭐ 추천
  [4] pulse: Max Peak =      0 ⭐ 추천
  [4] pulse: Max Peak =      0 ⭐ 추천
  [5] default: Max Peak =      0 ⭐ 추천

✅ 추천 장치: INDEX 3
   INPUT_DEVICE_INDEX = 3 로 설정 후 재시도하세요
  [5] default: Max Peak =      0 ⭐ 추천

✅ 추천 장치: INDEX 3
   INPUT_DEVICE_INDEX = 3 로 설정 후 재시도하세요


In [ ]:
import subprocess
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import os
import threading
from queue import Queue
import time

# ============================================================
# ✅ 개선된 솔루션: 지속적인 PulseAudio 스트림 사용
# ============================================================
print("=" * 70)
print("🎵 웹 브라우저 소리 녹음 (개선된 실시간 감지)")
print("=" * 70)

# ============================================================
# 1. PulseAudio 모니터 장치 찾기
# ============================================================
def find_pulseaudio_monitor():
    """PulseAudio 모니터 장치의 이름 찾기"""
    try:
        result = subprocess.run(['pactl', 'list', 'short', 'sources'],
                              capture_output=True, text=True, check=True)
        for line in result.stdout.strip().split('\n'):
            if line.strip():
                parts = line.split()
                source_name = parts[1]
                if 'monitor' in source_name.lower():
                    print(f"\n✅ 발견된 모니터 장치: {source_name}")
                    return source_name
    except Exception as e:
        print(f"❌ pactl 오류: {e}")
    return None

MONITOR_SOURCE = find_pulseaudio_monitor()

if not MONITOR_SOURCE:
    print("\n⚠️  모니터 장치를 찾을 수 없습니다!")
    print("다음 명령어를 터미널에서 실행하세요:")
    print("  pactl load-module module-loopback latency_msec=1 sink=null")
    print("\n그 후 아래 명령어로 장치를 확인하세요:")
    print("  pactl list short sources")
else:
    # ============================================================
    # 2. 설정
    # ============================================================
    THRESHOLD_PEAK = 800          # 녹음 시작 임계값 (증가시킴)
    SILENCE_LIMIT = 100.0           # 무음 지속 시간
    PAUSE_TOLERANCE_SECONDS = 1.5 # 짧은 무음 허용 (길게 설정)
    MIN_RECORDING_DURATION = 0.5  # 최소 녹음 길이
    
    OUTPUT_FORMAT = "mp3"
    MP3_BITRATE = "192k"
    CHUNK = 4096                  # 더 큰 청크로 정확한 감지
    
    # ============================================================
    # 3. PulseAudio에서 샘플 레이트 확인
    # ============================================================
    try:
        result = subprocess.run(['pactl', 'list', 'sources'],
                              capture_output=True, text=True, check=True)
        RATE = 48000  # 기본값
        for line in result.stdout.split('\n'):
            if MONITOR_SOURCE in line or 'Sample Specification' in line:
                if 'Hz' in line:
                    parts = line.split()
                    for part in parts:
                        if 'Hz' in part:
                            RATE = int(part.replace('Hz', ''))
                            print(f"📊 감지된 샘플레이트: {RATE}Hz")
                            break
    except:
        RATE = 48000

    CHANNELS = 2
    CHUNK_DURATION = CHUNK / RATE

    print(f"\n🎤 녹음 설정:")
    print(f"   - 소스: {MONITOR_SOURCE}")
    print(f"   - 샘플레이트: {RATE}Hz")
    print(f"   - 채널: {CHANNELS}")
    print(f"   - 청크 크기: {CHUNK}")
    print(f"   - 임계값: {THRESHOLD_PEAK}")
    print(f"   - 포맷: {OUTPUT_FORMAT}")

    # ============================================================
    # 4. 백그라운드 MP3 변환 스레드
    # ============================================================
    conversion_queue = Queue()
    keep_converting = True

    def mp3_converter_thread():
        while keep_converting:
            try:
                wav_file, mp3_file = conversion_queue.get(timeout=1)
                try:
                    print(f"\n🔄 MP3 변환 중: {os.path.basename(wav_file)}")
                    audio_data = AudioSegment.from_wav(wav_file)
                    audio_data.export(mp3_file, format="mp3", bitrate=MP3_BITRATE)
                    os.remove(wav_file)
                    file_size = os.path.getsize(mp3_file) / 1024
                    print(f"✅ 완료: {os.path.basename(mp3_file)} ({file_size:.1f}KB)")
                except Exception as e:
                    print(f"❌ 변환 실패: {e}")
                conversion_queue.task_done()
            except:
                continue

    converter = threading.Thread(target=mp3_converter_thread, daemon=True)
    converter.start()

    # ============================================================
    # 5. 무음 감지 설정 (사전 계산)
    # ============================================================
    PAUSE_TOLERANCE_CHUNKS = int(PAUSE_TOLERANCE_SECONDS / CHUNK_DURATION)
    SILENCE_THRESHOLD_CHUNKS = int(SILENCE_LIMIT / CHUNK_DURATION)
    
    print(f"\n📊 무음 감지 설정:")
    print(f"   - 짧은 무음 허용: {PAUSE_TOLERANCE_SECONDS}초 ({PAUSE_TOLERANCE_CHUNKS}청크)")
    print(f"   - 녹음 종료 무음: {SILENCE_LIMIT}초 ({SILENCE_THRESHOLD_CHUNKS}청크)")
    
    # ============================================================
    # 6. 지속적인 parec 스트림 사용
    # ============================================================
    print(f"\n🎵 준비 완료!")
    print(f"⏹️  웹 브라우저에서 소리를 재생하세요.")
    print(f"💻 종료: Ctrl+C 입력\n")
    print("=" * 70 + "\n")

    counter = 1
    recording_process = None
    parec_process = None
    silence_count = 0
    current_wav = None
    recording = False
    peak_history = []

    try:
        # 지속적인 parec 스트림 시작 (모니터링 전용)
        monitor_cmd = [
            'parec',
            '-d', MONITOR_SOURCE,
            '--format=s16',
            '--rate', str(RATE),
            '--channels', str(CHANNELS),
            '--latency=1'
        ]
        
        parec_process = subprocess.Popen(
            monitor_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            bufsize=0  # 버퍼링 안함
        )
        
        print("✅ PulseAudio 모니터링 스트림 시작됨\n")
        
        while True:
            try:
                # 지속적으로 오디오 데이터 읽기
                raw_data = parec_process.stdout.read(CHUNK * 2)
                
                if len(raw_data) > 0:
                    data = np.frombuffer(raw_data, dtype=np.int16)
                    peak = np.max(np.abs(data)) if len(data) > 0 else 0
                    peak_history.append(peak)
                    if len(peak_history) > 10:
                        peak_history.pop(0)
                else:
                    peak = 0
                
                # 평균 Peak 계산 (노이즈 감소)
                avg_peak = np.mean(peak_history) if peak_history else 0
                
                status = "🔴 녹음중" if recording else "⏳ 대기"
                queue_size = conversion_queue.qsize()
                print(f"{status} | Peak: {peak:5d} | 평균: {avg_peak:5.0f} | 임계값: {THRESHOLD_PEAK} | 대기: {queue_size} | 무음: {silence_count}", end='\r')
                
                # ==================== 녹음 시작 ====================
                if peak > THRESHOLD_PEAK and not recording:
                    recording = True
                    silence_count = 0
                    current_wav = f"record_{counter}_temp.wav"
                    
                    # ffmpeg로 WAV 파일로 녹음 시작 (동시에 진행)
                    ffmpeg_cmd = [
                        'ffmpeg',
                        '-f', 'pulse',
                        '-i', MONITOR_SOURCE,
                        '-acodec', 'pcm_s16le',
                        '-ar', str(RATE),
                        '-ac', str(CHANNELS),
                        '-loglevel', 'error',  # 에러만 출력
                        current_wav,
                        '-y'
                    ]
                    
                    recording_process = subprocess.Popen(
                        ffmpeg_cmd,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.PIPE
                    )
                    
                    print(f"\n✅ 녹음 시작: record_{counter}.{OUTPUT_FORMAT} (Peak: {peak})")
                
                # ==================== 녹음 중 ====================
                elif recording:
                    # 임계값 이상: 무음 카운터 초기화
                    if peak > THRESHOLD_PEAK:
                        silence_count = 0
                    # 임계값 이하: 무음 카운터 증가
                    else:
                        silence_count += 1
                    
                    # ==================== 녹음 종료 ====================
                    # 임계값 이하 상태가 (짧은 무음 허용 + 필요 무음)만큼 지속되면 종료
                    if silence_count > (PAUSE_TOLERANCE_CHUNKS + SILENCE_THRESHOLD_CHUNKS):
                        if recording_process:
                            recording_process.terminate()
                            try:
                                recording_process.wait(timeout=2)
                            except subprocess.TimeoutExpired:
                                recording_process.kill()
                        
                        # 파일 크기 확인
                        time.sleep(0.5)  # ffmpeg가 파일을 완성할 시간 제공
                        
                        if os.path.exists(current_wav):
                            file_size = os.path.getsize(current_wav)
                            
                            # 최소 크기 체크
                            min_bytes = int(MIN_RECORDING_DURATION * RATE * CHANNELS * 2)
                            
                            if file_size > min_bytes:
                                if OUTPUT_FORMAT == "mp3":
                                    mp3_file = f"record_{counter}.mp3"
                                    conversion_queue.put((current_wav, mp3_file))
                                    duration = file_size / (RATE * CHANNELS * 2)
                                    print(f"✔️  저장됨: {mp3_file} ({duration:.2f}초)")
                                else:
                                    wav_file = f"record_{counter}.wav"
                                    os.rename(current_wav, wav_file)
                                    duration = file_size / (RATE * CHANNELS * 2)
                                    print(f"✔️  저장됨: {wav_file} ({duration:.2f}초)")
                            else:
                                if os.path.exists(current_wav):
                                    os.remove(current_wav)
                                print(f"⚠️  녹음이 너무 짧음 ({file_size}/{min_bytes}바이트) - 무시됨")
                        
                        counter += 1
                        recording = False
                        silence_count = 0
                        current_wav = None
                        recording_process = None
                
            except Exception as e:
                print(f"\n오류: {e}")
                if recording_process:
                    recording_process.terminate()
                break

    except KeyboardInterrupt:
        print("\n\n⏹️  녹음 중지됨")
        
        if recording_process:
            recording_process.terminate()
            try:
                recording_process.wait(timeout=2)
            except subprocess.TimeoutExpired:
                recording_process.kill()
        
        if parec_process:
            parec_process.terminate()
            try:
                parec_process.wait(timeout=1)
            except subprocess.TimeoutExpired:
                parec_process.kill()
        
        print("⏳ 변환 완료 대기...")
        keep_converting = False
        conversion_queue.join()
        
    finally:
        if parec_process:
            try:
                parec_process.terminate()
                parec_process.wait(timeout=1)
            except:
                pass
        
        print("\n✅ 프로그램 종료")
        print("=" * 70)

🎵 웹 브라우저 소리 녹음 (개선된 실시간 감지)

✅ 발견된 모니터 장치: alsa_output.pci-0000_03_00.6.analog-stereo.monitor

🎤 녹음 설정:
   - 소스: alsa_output.pci-0000_03_00.6.analog-stereo.monitor
   - 샘플레이트: 48000Hz
   - 채널: 2
   - 청크 크기: 4096
   - 임계값: 800
   - 포맷: mp3

📊 무음 감지 설정:
   - 짧은 무음 허용: 1.5초 (17청크)
   - 녹음 종료 무음: 50.0초 (585청크)

🎵 준비 완료!
⏹️  웹 브라우저에서 소리를 재생하세요.
💻 종료: Ctrl+C 입력


✅ PulseAudio 모니터링 스트림 시작됨

⏳ 대기 | Peak:   846 | 평균:   614 | 임계값: 800 | 대기: 0 | 무음: 0
✅ 녹음 시작: record_1.mp3 (Peak: 846)
⏳ 대기 | Peak:   846 | 평균:   614 | 임계값: 800 | 대기: 0 | 무음: 0 1
✅ 녹음 시작: record_1.mp3 (Peak: 846)
✔️  저장됨: record_1.mp3 (123.90초)9 | 임계값: 800 | 대기: 0 | 무음: 602
🔄 MP3 변환 중: record_1_temp.wav

✔️  저장됨: record_1.mp3 (123.90초)| 임계값: 800 | 대기: 0 | 무음: 0
🔄 MP3 변환 중: record_1_temp.wav

⏳ 대기 | Peak:  1307 | 평균:   176 | 임계값: 800 | 대기: 0 | 무음: 0
✅ 녹음 시작: record_2.mp3 (Peak: 1307)
⏳ 대기 | Peak:  1307 | 평균:   176 | 임계값: 800 | 대기: 0 | 무음: 0 0
✅ 녹음 시작: record_2.mp3 (Peak: 1307)
✅ 완료: record_1.mp3 (2905.4KB)436 | 임계값: 800 | 대기: 0 | 무음: